# 05b - Game Predictions (Binary Ensemble)

Predict full game statlines for starting pitchers using the binary model ensemble.

**Key differences from 05:**
- Uses 7 separate binary classifiers instead of one 7-class model
- Each outcome model has its own optimized features
- Should have better calibration for elite pitchers

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
from datetime import date, timedelta
from pathlib import Path

from src.game_predictor_binary import GamePredictorBinary, format_prediction_summary
from src.data.mlb_api import get_games_with_lineups, check_lineup_status

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

/Applications/miniconda3/envs/mlbenv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Initialize Predictor

In [2]:
# Initialize predictor with binary ensemble
predictor = GamePredictorBinary(
    ensemble_dir='../models/binary_ensemble',
    preprocessor_path='../models/matchup_preprocessor.pkl',
    pitcher_profiles_path='../data/profiles/pitcher_arsenal.csv',
    batter_profiles_path='../data/profiles/batter_profiles.csv',
    pitcher_rolling_path='../data/profiles/pitcher_rolling.csv',
    batter_rolling_path='../data/profiles/batter_rolling.csv',
    park_factors_path= "../data/profiles/park_factors.csv"
)

print(f"Feature names: {len(predictor.feature_names)}")
print(f"Outcome classes: {predictor.outcome_classes}")

Feature names: 207
Outcome classes: ['1B', '2B', '3B', 'BB', 'HR', 'K', 'OUT']


## 2. Check Today's Games

In [3]:
# Check lineup availability
today = date.today().strftime("%Y-%m-%d")
print(f"Checking games for {today}...")

status = check_lineup_status(today)
print(f"\nTotal games: {status['total_games']}")
print(f"With lineups: {status['games_with_lineups']}")
print(f"With probable pitchers: {status['games_with_probable_pitchers']}")

print("\nGames:")
for game in status["games"]:
    away = game["away_team"]["abbreviation"]
    home = game["home_team"]["abbreviation"]
    away_p = game["away_pitcher"]["pitcher_name"] if game["away_pitcher"] else "TBD"
    home_p = game["home_pitcher"]["pitcher_name"] if game["home_pitcher"] else "TBD"
    lineups = "Lineups" if game["lineups_available"] else "No lineups"
    print(f"  {away} @ {home}: {away_p} vs {home_p} [{lineups}]")

Checking games for 2026-04-05...

Total games: 16
With lineups: 9
With probable pitchers: 16

Games:
  CHC @ CLE: Edward Cabrera vs Slade Cecconi [Lineups]
  CHC @ CLE: Shota Imanaga vs Parker Messick [No lineups]
  LAD @ WSH: Roki Sasaki vs Foster Griffin [Lineups]
  BAL @ PIT: Chris Bassitt vs Braxton Ashcraft [Lineups]
  SD @ BOS: Walker Buehler vs Ranger Suarez [Lineups]
  MIA @ NYY: Chris Paddack vs Max Fried [Lineups]
  TB @ MIN: Nick Martinez vs Simeon Woods Richardson [Lineups]
  TOR @ CWS: Eric Lauer vs Davis Martin [Lineups]
  MIL @ KC: Kyle Harrison vs Kris Bubic [No lineups]
  CIN @ TEX: Chase Burns vs Jack Leiter [Lineups]
  PHI @ COL: Taijuan Walker vs Tomoyuki Sugano [Lineups]
  HOU @ ATH: Lance McCullers Jr. vs Jacob Lopez [No lineups]
  NYM @ SF: Kodai Senga vs Logan Webb [No lineups]
  SEA @ LAA: Luis Castillo vs Ryan Johnson [No lineups]
  ATL @ AZ: Martín Pérez vs Brandon Pfaadt [No lineups]
  STL @ DET: Kyle Leahy vs Keider Montero [No lineups]


## 3. Run Predictions

In [4]:
# Predict all games for today
predictions = predictor.predict_day(today)

print(f"\n{'='*60}")
print(f"PREDICTIONS FOR {today}")
print(f"{'='*60}")

for game in predictions:
    print(f"\n{game['away_team']} @ {game['home_team']} ({game['status']})")
    print("-" * 40)
    
    if game["away_prediction"]:
        print(format_prediction_summary(game["away_prediction"]))
    
    if game["home_prediction"]:
        print(format_prediction_summary(game["home_prediction"]))
    
    if game["errors"]:
        print(f"  Errors: {game['errors']}")


PREDICTIONS FOR 2026-04-05

CHC @ CLE (complete)
----------------------------------------
Edward Cabrera
  Target IP: 5.3, Sim IP: 5.3
  K: 5.5, BB: 2.5, H: 5.2, HR: 0.6
  Expected Runs: 2.28
  ERA (this start): 3.85
Slade Cecconi
  Target IP: 5.3, Sim IP: 5.3
  K: 4.5, BB: 1.6, H: 5.7, HR: 0.8
  Expected Runs: 2.36
  ERA (this start): 3.98

CHC @ CLE (awaiting_lineups)
----------------------------------------

LAD @ WSH (complete)
----------------------------------------
Roki Sasaki
  Target IP: 5.0, Sim IP: 5.0
  K: 3.8, BB: 2.8, H: 5.3, HR: 0.6
  Expected Runs: 2.52
  ERA (this start): 4.54
Foster Griffin
  Target IP: 4.9, Sim IP: 5.0
  K: 4.3, BB: 2.5, H: 6.1, HR: 1.0
  Expected Runs: 3.06
  ERA (this start): 5.51

BAL @ PIT (complete)
----------------------------------------
Chris Bassitt
  Target IP: 5.1, Sim IP: 5.0
  K: 5.2, BB: 1.4, H: 5.1, HR: 0.7
  Expected Runs: 2.10
  ERA (this start): 3.77
Braxton Ashcraft
  Target IP: 4.7, Sim IP: 4.7
  K: 5.2, BB: 1.8, H: 4.7, HR: 0.9


## 4. Detailed View - Single Game

In [5]:
# Show detailed predictions for first complete game
complete_games = [g for g in predictions if g['status'] == 'complete']

if complete_games:
    game = complete_games[0]
    print(f"Detailed view: {game['away_team']} @ {game['home_team']}")
    print("=" * 60)
    
    for side, pred in [("AWAY", game["away_prediction"]), ("HOME", game["home_prediction"])]:
        if pred:
            print(f"\n{side}: {pred['pitcher_name']}")
            print(f"Expected BF: {pred['expected_bf']}")
            print(f"\nExpected Stats:")
            for stat, val in pred['expected_stats'].items():
                print(f"  {stat}: {val}")
            
            print(f"\nPer-Batter Predictions (first time through):")
            for bp in pred['batter_predictions'][:9]:
                k_prob = bp['probabilities']['K']
                bb_prob = bp['probabilities']['BB']
                hr_prob = bp['probabilities']['HR']
                print(f"  {bp['batter_name'][:20]:20s}: K={k_prob:.1%}, BB={bb_prob:.1%}, HR={hr_prob:.1%}")
else:
    print("No complete games available")

Detailed view: CHC @ CLE

AWAY: Edward Cabrera
Expected BF: 23.0

Expected Stats:
  K: 5.54
  BB: 2.48
  H: 5.23
  1B: 3.56
  2B: 0.98
  3B: 0.09
  HR: 0.6
  ER: 2.28
  IP_approx: 5.3
  ERA_approx: 3.85

Per-Batter Predictions (first time through):
  Steven Kwan         : K=11.4%, BB=14.1%, HR=1.1%
  Chase DeLauter      : K=20.3%, BB=12.6%, HR=4.6%
  José Ramírez        : K=13.1%, BB=9.1%, HR=3.9%
  Kyle Manzardo       : K=29.7%, BB=9.7%, HR=3.9%
  Bo Naylor           : K=24.6%, BB=12.3%, HR=3.0%
  Daniel Schneemann   : K=27.2%, BB=12.1%, HR=2.6%
  Brayan Rocchio      : K=21.6%, BB=8.1%, HR=1.5%
  Gabriel Arias       : K=34.2%, BB=6.5%, HR=2.6%
  CJ Kayfus           : K=27.4%, BB=9.7%, HR=2.3%

HOME: Slade Cecconi
Expected BF: 23.0

Expected Stats:
  K: 4.55
  BB: 1.59
  H: 5.65
  1B: 3.51
  2B: 1.19
  3B: 0.14
  HR: 0.81
  ER: 2.36
  IP_approx: 5.3
  ERA_approx: 3.98

Per-Batter Predictions (first time through):
  Michael Busch       : K=24.0%, BB=9.4%, HR=5.5%
  Alex Bregman        :

## 5. Compare with Yesterday (Completed Games)

In [6]:
# Get yesterday's games (should have lineups from boxscore)
yesterday = (date.today() - timedelta(days=1)).strftime("%Y-%m-%d")
print(f"Predictions for yesterday ({yesterday}):")

yesterday_preds = predictor.predict_day(yesterday)

for game in yesterday_preds:
    if game['status'] == 'complete':
        print(f"\n{game['away_team']} @ {game['home_team']}")
        
        if game["away_prediction"]:
            p = game["away_prediction"]
            s = p["expected_stats"]
            print(f"  {p['pitcher_name']}: {s['K']:.1f} K, {s['BB']:.1f} BB, {s['H']:.1f} H, ERA={s['ERA_approx']}")
        
        if game["home_prediction"]:
            p = game["home_prediction"]
            s = p["expected_stats"]
            print(f"  {p['pitcher_name']}: {s['K']:.1f} K, {s['BB']:.1f} BB, {s['H']:.1f} H, ERA={s['ERA_approx']}")

Predictions for yesterday (2026-04-04):

STL @ DET
  Dustin May: 4.3 K, 2.4 BB, 5.4 H, ERA=5.1
  Jack Flaherty: 5.1 K, 1.9 BB, 4.8 H, ERA=3.92

TOR @ CWS
  Mason Fluharty: 0.3 K, 0.1 BB, 0.3 H, ERA=3.45
  Grant Taylor: 1.0 K, 0.4 BB, 0.4 H, ERA=1.59

MIL @ KC
  Chad Patrick: 3.4 K, 1.2 BB, 4.9 H, ERA=4.24
  Luinder Avila: 3.5 K, 2.5 BB, 5.3 H, ERA=5.71

MIL @ KC
  Logan Henderson: 3.2 K, 1.3 BB, 4.6 H, ERA=4.57
  Seth Lugo: 5.2 K, 1.7 BB, 5.2 H, ERA=3.35

LAD @ WSH
  Tyler Glasnow: 6.8 K, 1.7 BB, 4.4 H, ERA=2.78
  Jake Irvin: 3.8 K, 2.3 BB, 6.1 H, ERA=6.76

HOU @ ATH
  Tatsuya Imai: 5.4 K, 3.6 BB, 4.6 H, ERA=6.51
  Luis Morales: 4.5 K, 2.5 BB, 5.3 H, ERA=4.47

BAL @ PIT
  Shane Baz: 5.8 K, 2.0 BB, 5.4 H, ERA=4.92
  Carmen Mlodzinski: 4.8 K, 2.0 BB, 4.2 H, ERA=3.19

SD @ BOS
  Randy Vásquez: 3.9 K, 1.7 BB, 6.1 H, ERA=6.46
  Connelly Early: 4.2 K, 2.0 BB, 4.6 H, ERA=4.45

CIN @ TEX
  Rhett Lowder: 3.9 K, 2.5 BB, 6.1 H, ERA=5.98
  Kumar Rocker: 4.9 K, 2.4 BB, 5.0 H, ERA=4.66

MIA @ NYY
  

## 6. Elite Pitcher Check

Verify predictions for elite K pitchers are improved.

In [7]:
# Find elite pitchers in recent predictions
all_preds = yesterday_preds + predictions

elite_k_threshold = 7.0  # Expected Ks
elite_pitchers = []

for game in all_preds:
    for pred_key in ["away_prediction", "home_prediction"]:
        pred = game.get(pred_key)
        if pred and pred["expected_stats"]["K"] >= elite_k_threshold:
            elite_pitchers.append({
                "name": pred["pitcher_name"],
                "K": pred["expected_stats"]["K"],
                "BB": pred["expected_stats"]["BB"],
                "H": pred["expected_stats"]["H"],
                "ERA": pred["expected_stats"]["ERA_approx"],
                "BF": pred["expected_bf"],
            })

if elite_pitchers:
    print(f"Elite K predictions (>= {elite_k_threshold} K):")
    elite_df = pd.DataFrame(elite_pitchers).sort_values("K", ascending=False)
    print(elite_df.to_string(index=False))
else:
    print("No elite K predictions found in recent games")

No elite K predictions found in recent games


## 7. Summary Table

In [8]:
# Create summary DataFrame
rows = []

for game in predictions:
    for side, pred_key in [("away", "away_prediction"), ("home", "home_prediction")]:
        pred = game.get(pred_key)
        if pred:
            s = pred["expected_stats"]
            rows.append({
                "Game": f"{game['away_team']}@{game['home_team']}",
                "Pitcher": pred["pitcher_name"],
                "BF": pred["expected_bf"],
                "K": s["K"],
                "BB": s["BB"],
                "H": s["H"],
                "HR": s["HR"],
                "ER": s["ER"],
                "IP": s["IP_approx"],
                "ERA": s["ERA_approx"],
            })

if rows:
    summary_df = pd.DataFrame(rows)
    print(f"\nPrediction Summary for {today}:")
    print(summary_df.to_string(index=False))
else:
    print("No predictions available")


Prediction Summary for 2026-04-05:
   Game                 Pitcher   BF    K   BB    H   HR   ER  IP  ERA
CHC@CLE          Edward Cabrera 23.0 5.54 2.48 5.23 0.60 2.28 5.3 3.85
CHC@CLE           Slade Cecconi 23.0 4.55 1.59 5.65 0.81 2.36 5.3 3.98
LAD@WSH             Roki Sasaki 21.5 3.76 2.84 5.34 0.60 2.52 5.0 4.54
LAD@WSH          Foster Griffin 21.0 4.32 2.48 6.10 0.98 3.06 5.0 5.51
BAL@PIT           Chris Bassitt 22.0 5.25 1.41 5.10 0.66 2.10 5.0 3.77
BAL@PIT        Braxton Ashcraft 20.0 5.21 1.82 4.67 0.92 2.36 4.7 4.55
 SD@BOS          Walker Buehler 23.0 5.13 1.69 5.02 0.80 2.49 5.3 4.20
 SD@BOS           Ranger Suarez 24.0 4.72 1.90 5.86 0.75 2.57 5.7 4.09
MIA@NYY           Chris Paddack 21.0 5.20 1.99 5.42 1.22 2.96 5.0 5.33
MIA@NYY               Max Fried 25.5 5.59 1.65 5.88 0.49 2.01 6.0 3.01
 TB@MIN           Nick Martinez 24.0 6.05 1.39 5.05 1.00 2.42 5.7 3.84
 TB@MIN Simeon Woods Richardson 21.0 3.76 1.74 5.30 0.61 2.33 5.0 4.19
TOR@CWS              Eric Lauer 21.5 4.98